Azure offers a full container platform across multiple abstraction levels:

- **ACR (Azure Container Registry)** — store and manage container images
- **ACI (Azure Container Instances)** — run a container directly, no orchestration needed
- **AKS (Azure Kubernetes Service)** — production-grade Kubernetes with a managed control plane
- **Azure Container Apps** — managed Kubernetes-based platform that hides cluster complexity

Choosing between them is primarily about how much control and complexity you want to manage.

## Containers vs VMs vs Serverless

```
Virtual Machine          Container               Serverless (Functions)
┌──────────────┐        ┌──────────────┐        ┌──────────────┐
│  App + Libs  │        │  App + Libs  │        │  Function    │
│  Guest OS    │        │  (no OS)     │        │              │
│  Hypervisor  │        │  Container   │        │  Managed     │
│  Host OS     │        │  Runtime     │        │  Runtime     │
│  Hardware    │        │  Host OS     │        │  Hardware    │
└──────────────┘        │  Hardware    │        └──────────────┘
                        └──────────────┘
Slowest to start        Fast (seconds)          Fastest (ms)
Full OS control         Shared kernel           No management
Heaviest image          Lightweight image       No image needed
```

Containers are ideal for: microservices, long-running processes, workloads you want to run identically in dev and prod, and anything you are migrating from on-premises Docker environments.

## Azure Container Registry (ACR)

**ACR** is Azure's managed private Docker container registry — the equivalent of AWS ECR. It stores and distributes container images and Helm charts, integrated with Entra ID and Azure RBAC.

### SKUs

| SKU | Storage | Throughput | Key features |
|---|---|---|---|
| **Basic** | 10 GB | Low | Dev/test; manual tasks |
| **Standard** | 100 GB | Medium | Production workloads |
| **Premium** | 500 GB | High | Geo-replication, private endpoints, customer-managed keys, dedicated data endpoints |

### Authentication

| Method | When to use |
|---|---|
| **Managed Identity** | AKS, ACI, App Service — no credentials, recommended |
| **Service Principal** | CI/CD pipelines — GitHub Actions, Azure DevOps |
| **Admin user** | Quick testing only — single username/password, not recommended for production |
| **Entra ID token** | Interactive developer use via `az acr login` |

### Common CLI commands

```bash
# Create a registry
az acr create --name myregistry --resource-group my-rg --sku Premium

# Authenticate Docker to ACR
az acr login --name myregistry

# Tag and push an image
docker tag my-app:latest myregistry.azurecr.io/my-app:latest
docker push myregistry.azurecr.io/my-app:latest

# List images in the registry
az acr repository list --name myregistry

# Build an image in ACR (no local Docker needed)
az acr build --registry myregistry --image my-app:latest .
```

### ACR Tasks

**ACR Tasks** build container images in the cloud without a local Docker installation:
- **Quick task** — `az acr build` — single on-demand build
- **Triggered task** — rebuild automatically on Git commit or base image update
- **Scheduled task** — rebuild on a cron schedule (e.g. nightly base image refresh)

### Geo-replication (Premium)

Replicate the registry to multiple Azure regions — AKS clusters in each region pull images from a local replica, reducing latency and egress costs. Replication is automatic and transparent to clients.

## Azure Container Instances (ACI)

**ACI** is the fastest way to run a container in Azure — no clusters, no orchestration, no node management. You describe what you want to run and Azure starts it in seconds.

ACI is best for:
- Simple one-off tasks and batch jobs
- CI/CD pipeline build agents
- Dev/test environments
- Burst capacity from AKS (Virtual Nodes)

### Container Groups

The top-level resource in ACI is a **container group** — a collection of containers that share a lifecycle, network, and storage. This is equivalent to a Kubernetes Pod.

```
Container Group
├── Container A (web server)     ← share localhost network
├── Container B (log sidecar)    ← share mounted volume
└── Init Container (setup)       ← runs to completion before A and B start
```

Containers within a group share:
- The same IP address and port namespace — communicate over `localhost`
- Mounted volumes (Azure Files, empty dir, git repo clone)

### Networking

| Mode | Description |
|---|---|
| **Public IP** | Azure assigns a public IP and optional DNS label — `mycontainer.eastus.azurecontainer.io` |
| **VNet injection** | Deploy the container group into a subnet — private IP only, accessible within the VNet |

### Restart policy

| Policy | Behaviour | Use case |
|---|---|---|
| **Always** | Restart on exit | Long-running services |
| **OnFailure** | Restart only if exit code ≠ 0 | Batch jobs that may succeed |
| **Never** | Never restart | One-off tasks, run-to-completion |

```bash
# Run a container directly from ACR
az container create \
  --resource-group my-rg \
  --name my-job \
  --image myregistry.azurecr.io/my-app:latest \
  --cpu 1 --memory 1.5 \
  --restart-policy Never \
  --environment-variables ENV=prod \
  --assign-identity   # system-assigned managed identity
```

## Azure Kubernetes Service (AKS)

**AKS** is Azure's managed Kubernetes service. Azure manages the **control plane** (API server, etcd, scheduler, controller manager) at no cost — you pay only for the worker node VMs.

```
AKS Control Plane (Azure managed, free)
  ├── API Server
  ├── etcd (state store)
  └── Scheduler / Controller Manager
            │
            ▼
Node Pools (your subscription — you pay for VMs)
  ├── System node pool  (runs AKS system pods — kube-dns, metrics-server)
  └── User node pool(s) (runs your application workloads)
```

### Why Kubernetes / When to choose AKS

| Reason | Detail |
|---|---|
| **Portability** | Kubernetes is cloud-agnostic — same YAML manifests on Azure, AWS, GCP, on-prem |
| **Existing K8s workloads** | Lift and shift without rewriting |
| **Rich ecosystem** | Helm charts, service meshes (Istio, Linkerd), KEDA, Dapr |
| **Advanced scheduling** | Node affinity, taints/tolerations, pod topology spread |
| **Large microservices fleet** | Hundreds of services with complex dependencies |

## AKS Node Pools

A **node pool** is a group of VMs with the same size and configuration. Every AKS cluster has at least one **system node pool** and optionally one or more **user node pools**.

| Pool type | Purpose |
|---|---|
| **System** | Runs AKS system components (DNS, metrics); required; tainted to repel app workloads |
| **User** | Runs your application pods; add as many as needed |

### Multiple node pools — why they matter

```
AKS Cluster
├── System pool    Standard_D2s_v5  (2 nodes — cluster infrastructure)
├── General pool   Standard_D4s_v5  (autoscales 3–20 — app workloads)
├── GPU pool       Standard_NC6s_v3 (0–5 — ML inference pods)
└── Spot pool      Standard_D8s_v5  Spot (0–50 — batch/CI workloads)
```

- Different VM sizes per pool — GPU nodes only for GPU workloads
- Spot node pools for cost-sensitive, fault-tolerant workloads
- Independent scaling per pool
- OS type per pool: Linux or Windows

### Virtual Nodes (ACI-backed)

A **Virtual Node** is a special node pool backed by ACI rather than VMs. Pods scheduled onto virtual nodes run as ACI container instances — they start in seconds and require no pre-provisioned capacity.

Use virtual nodes for burst workloads: normal pods run on VM node pools; burst pods spill onto ACI. This eliminates the need to pre-provision capacity for peak load.

## AKS Networking

### Network plugins

| Plugin | How pods get IPs | Best for |
|---|---|---|
| **Kubenet** | Pods get IPs from a private overlay network; nodes get VNet IPs | Small clusters; conserves VNet IP space |
| **Azure CNI** | Every pod gets a real VNet IP — directly routable | Production; needed for VNet peering, private endpoints |
| **Azure CNI Overlay** | Pods get overlay IPs but use Azure CNI for VNet connectivity | Large clusters needing Azure CNI features with less IP consumption |
| **Azure CNI Powered by Cilium** | eBPF-based dataplane — high performance, network policy | Performance-critical workloads |

### Ingress

To expose services externally, AKS uses an **Ingress controller**:

| Option | Description |
|---|---|
| **NGINX Ingress Controller** | Community standard; deploy via Helm |
| **Azure Application Gateway Ingress Controller (AGIC)** | Uses Azure Application Gateway as the ingress — WAF included |
| **Azure API Management** | Enterprise API gateway in front of AKS services |

### Load balancer integration

When you create a Kubernetes Service of type `LoadBalancer`, AKS automatically provisions an **Azure Load Balancer** and assigns a public or internal IP.

## AKS Scaling

AKS supports scaling at two levels — pods and nodes.

### Pod scaling

| Scaler | What it scales | How |
|---|---|---|
| **HPA (Horizontal Pod Autoscaler)** | Number of pod replicas | Based on CPU, memory, or custom metrics |
| **VPA (Vertical Pod Autoscaler)** | CPU/memory requests on pods | Recommends or automatically adjusts resource requests |
| **KEDA** | Pod replicas to zero and back | Event-driven — queue length, Cosmos DB, HTTP request count |

**KEDA (Kubernetes Event-Driven Autoscaling)** is built into AKS and lets pods scale to zero — similar to Azure Functions on the Consumption plan, but for containerised workloads.

### Node scaling

The **Cluster Autoscaler** automatically adds or removes nodes when:
- Pods cannot be scheduled due to insufficient node capacity → **scale out** (add nodes)
- Nodes are underutilised and pods can be rescheduled elsewhere → **scale in** (remove nodes)

```
Configure per node pool:
min-count: 2    max-count: 20    enable-cluster-autoscaler: true
```

### AKS Automatic (preview)

**AKS Automatic** is a new mode where Azure manages node provisioning, scaling, upgrades, and security configuration entirely — you only manage workloads. It is the closest AKS equivalent to a fully managed Kubernetes experience.

## AKS Integrations

### ACR integration

Attach an ACR to AKS with one command — Azure grants the AKS Managed Identity the `AcrPull` role, so nodes can pull images without credentials:

```bash
az aks update --name my-cluster --resource-group my-rg --attach-acr myregistry
```

### Entra ID and Kubernetes RBAC

AKS integrates with Entra ID for authentication. Users authenticate to the Kubernetes API server with their Entra ID token. Map Entra ID users and groups to Kubernetes RBAC roles for a unified identity model.

### Azure Key Vault — Secrets Store CSI Driver

Mount Key Vault secrets, certificates, and keys as files or environment variables in pods — using the pod's Managed Identity, no credentials stored in Kubernetes Secrets:

```yaml
volumes:
  - name: secrets
    csi:
      driver: secrets-store.csi.k8s.io
      readOnly: true
      volumeAttributes:
        secretProviderClass: my-key-vault-provider
```

### Azure Monitor and Container Insights

Enable **Container Insights** to collect logs and metrics from all pods and nodes into a Log Analytics workspace — CPU, memory, disk, network per pod, plus application logs via stdout/stderr.

## AKS Upgrades

AKS manages Kubernetes version upgrades for both the control plane and node pools.

### Upgrade process

```
1. Upgrade control plane   → API server updated (brief API unavailability)
2. Upgrade node pools      → one node at a time: cordon → drain → reimage → uncordon
```

During node upgrade, AKS cordons the node (no new pods scheduled), drains running pods to other nodes, reimages the node with the new OS, then brings it back online.

### Upgrade channels

| Channel | Behaviour |
|---|---|
| **node-image** | Automatically update node OS images (security patches) without changing K8s version |
| **patch** | Auto-upgrade to latest patch within the current minor version |
| **stable** | Auto-upgrade to latest stable Kubernetes minor version |
| **rapid** | Auto-upgrade to latest Kubernetes version as soon as it ships |
| **none** | No automatic upgrades — you control all upgrades manually |

> Use **node-image** channel as a minimum for all clusters — OS security patches applied automatically without Kubernetes version risk.

## Azure Container Apps

**Azure Container Apps** is a managed platform built on Kubernetes and KEDA that abstracts away cluster management entirely. You deploy containers without writing Kubernetes YAML or managing node pools.

| | ACI | Container Apps | AKS |
|---|---|---|---|
| Kubernetes | No | Hidden (managed) | Full access |
| Scale to zero | Yes | Yes (KEDA-based) | Yes (KEDA) |
| Ingress / routing | Basic | Built-in HTTP routing | Ingress controller |
| VNet integration | Yes | Yes | Yes |
| Dapr integration | No | Built-in | Via sidecar |
| Best for | One-off jobs | Microservices, event-driven apps | Full Kubernetes control |

Container Apps includes built-in **Dapr** (Distributed Application Runtime) support — service discovery, pub/sub, state management, and secret access via a sidecar, with no SDK changes to your application code.

## Decision Guide

| Scenario | Use |
|---|---|
| Run a one-off job or batch task | ACI |
| Burst capacity from AKS | ACI via Virtual Nodes |
| Simple web app or API | App Service or Container Apps |
| Event-driven microservices, scale to zero | Container Apps (KEDA) |
| Existing Kubernetes workloads, full K8s control | AKS |
| ML training / GPU workloads | AKS (GPU node pool) |
| Multi-cloud or on-prem K8s portability | AKS |
| Private image registry for any of the above | ACR |

## Working with ACR, ACI, and AKS via Python

In [ ]:
# pip install azure-identity azure-mgmt-containerregistry azure-mgmt-containerinstance azure-mgmt-containerservice

In [ ]:
from azure.identity import DefaultAzureCredential
from azure.mgmt.containerregistry import ContainerRegistryManagementClient
from azure.mgmt.containerinstance import ContainerInstanceManagementClient
from azure.mgmt.containerservice import ContainerServiceClient

credential = DefaultAzureCredential()
subscription_id = "<your-subscription-id>"
resource_group  = "my-rg"

acr_client = ContainerRegistryManagementClient(credential, subscription_id)
aci_client = ContainerInstanceManagementClient(credential, subscription_id)
aks_client = ContainerServiceClient(credential, subscription_id)

In [ ]:
# List all ACR registries and their SKUs
for registry in acr_client.registries.list():
    print(f"{registry.name:<30} SKU: {registry.sku.name:<10} "
          f"Login: {registry.login_server}")

In [ ]:
from azure.mgmt.containerinstance.models import (
    ContainerGroup, Container, ResourceRequirements,
    ResourceRequests, OperatingSystemTypes, ContainerGroupRestartPolicy
)

# Create a container group (ACI)
container_group = ContainerGroup(
    location="eastus",
    containers=[
        Container(
            name="my-job",
            image="myregistry.azurecr.io/my-app:latest",
            resources=ResourceRequirements(
                requests=ResourceRequests(cpu=1.0, memory_in_gb=1.5)
            ),
            environment_variables=[]
        )
    ],
    os_type=OperatingSystemTypes.LINUX,
    restart_policy=ContainerGroupRestartPolicy.NEVER
)

poller = aci_client.container_groups.begin_create_or_update(
    resource_group, "my-job-group", container_group
)
cg = poller.result()
print(f"Container group: {cg.name}  State: {cg.provisioning_state}")

In [ ]:
# List all AKS clusters in the subscription
for cluster in aks_client.managed_clusters.list():
    print(f"{cluster.name:<30} K8s: {cluster.kubernetes_version:<8} "
          f"Nodes: {sum(p.count for p in cluster.agent_pool_profiles or [])} "
          f"Status: {cluster.provisioning_state}")

In [ ]:
# List node pools in an AKS cluster
cluster_name = "my-aks-cluster"
pools = aks_client.agent_pools.list(resource_group, cluster_name)
for pool in pools:
    print(f"  {pool.name:<20} Size: {pool.vm_size:<25} "
          f"Count: {pool.count}  Mode: {pool.mode}")

In [ ]:
import subprocess

# Get AKS credentials (writes to ~/.kube/config)
subprocess.run([
    "az", "aks", "get-credentials",
    "--resource-group", resource_group,
    "--name", cluster_name,
    "--overwrite-existing"
], check=True)

# Verify connection
result = subprocess.run(["kubectl", "get", "nodes"], capture_output=True, text=True)
print(result.stdout)

## Summary

| Concept | Key Takeaway |
|---|---|
| ACR | Private managed image registry; SKUs: Basic/Standard/Premium; geo-replication on Premium |
| ACR Tasks | Build images in the cloud — quick, triggered, or scheduled |
| ACR auth | Managed Identity for Azure services; Service Principal for CI/CD |
| ACI | Fastest way to run a container — no cluster; groups share network + storage |
| ACI restart policy | Always (service), OnFailure (batch), Never (one-off) |
| AKS control plane | Managed by Azure at no cost — you pay only for node VMs |
| Node pools | System pool (required) + User pools; different sizes, OS, Spot per pool |
| Virtual Nodes | ACI-backed burst capacity — pods start in seconds, no pre-provisioned VMs |
| AKS networking | Kubenet (overlay) vs Azure CNI (real VNet IPs) — CNI for production |
| Cluster Autoscaler | Adds/removes nodes based on pod scheduling pressure |
| KEDA | Scale pods to zero and back based on event sources (queue depth, HTTP requests) |
| AKS + ACR | One command to grant AcrPull — no image pull credentials needed |
| AKS upgrades | node-image channel for OS patches; control plane + node pools upgraded separately |
| Container Apps | Managed K8s platform — KEDA scaling, Dapr built-in, no cluster to manage |
| Decision | ACI → one-off jobs; Container Apps → microservices; AKS → full Kubernetes control |